# Installations, imports, utils

In [ ]:
%%capture
!pip install einops langchain langchain_community xformers bitsandbytes sentence_transformers chromadb 
!pip install transformers torch torchvision -U

In [1]:
import sys
from torch import cuda, bfloat16
import torch
import transformers
from transformers import AutoTokenizer
from time import time
#import chromadb
#from chromadb.config import Settings
from langchain.llms import HuggingFacePipeline
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.vectorstores import Chroma

# Initialize model, tokenizer, query pipeline

Define the model, the device, and the `bitsandbytes` configuration.

In [2]:
model_id = '/kaggle/input/deepseek-r1/transformers/deepseek-r1-distill-qwen-14b/2'

device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'

# set quantization configuration to load large model with less GPU memory
# this requires the `bitsandbytes` library
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16
)

print(device)

cuda:0


Prepare the model and the tokenizer.

In [3]:
time_start = time()
model_config = transformers.AutoConfig.from_pretrained(
   model_id,
    trust_remote_code=True,
    max_new_tokens=1024
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    config=model_config,
    quantization_config=bnb_config,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
time_end = time()
print(f"Prepare model, tokenizer: {round(time_end-time_start, 3)} sec.")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Prepare model, tokenizer: 159.208 sec.


Netx, we define the query pipeline.  
In order to work correctly when we will define the HuggingFace pipeline, we will need to define here the max_length (to avoid falling back on the very short default length of `20`.

In [4]:
time_start = time()
query_pipeline = transformers.pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        torch_dtype=torch.float16,
        max_length=1024,
        device_map="auto",)
time_end = time()
print(f"Prepare pipeline: {round(time_end-time_start, 3)} sec.")

Device set to use cuda:0


Prepare pipeline: 0.098 sec.


We define a function for testing the pipeline.

In [5]:
def test_model(tokenizer, pipeline, message):
    """
    Perform a query
    print the result
    Args:
        tokenizer: the tokenizer
        pipeline: the pipeline
        message: the prompt
    Returns
        None
    """    
    time_start = time()
    sequences = pipeline(
        message,
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_length=200,)
    time_end = time()
    total_time = f"{round(time_end-time_start, 3)} sec."
    
    question = sequences[0]['generated_text'][:len(message)]
    answer = sequences[0]['generated_text'][len(message):]
    
    return f"Question: {question}\nAnswer: {answer}\nTotal time: {total_time}"


In [6]:
def test_model(tokenizer, pipeline, message):
    """
    执行查询并打印结果
    Args:
        tokenizer: 分词器
        pipeline: 管道流
        message: 提示词
    Returns
        None
    """    
    time_start = time()
    sequences = pipeline(
        message,
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_length=200,)
    time_end = time()
    total_time = f"{round(time_end-time_start, 3)} sec."
    
    question = sequences[0]['generated_text'][:len(message)]
    answer = sequences[0]['generated_text'][len(message):]
    
    return f"Question: {question}\nAnswer: {answer}\nTotal time: {total_time}"


## Test the query pipeline

We test the pipeline with few queries about European Union Artificial Intelligence Act (EU AI Act).

We also define here an utility function. This function will be used to display the output from the answer of the LLM.  
We include the calculation time, the question and the answer, formated so that will be easy to recognise them.

In [7]:
from IPython.display import display, Markdown
def colorize_text(text):
    for word, color in zip(["Reasoning", "Question", "Answer", "Total time"], ["blue", "red", "green", "magenta"]):
        text = text.replace(f"{word}:", f"\n\n**<font color='{color}'>{word}:</font>**")
    return text

Let's test now the pipeline with few queries.

In [8]:
response = test_model(tokenizer,
                    query_pipeline,
                   "人工智能法案")
display(Markdown(colorize_text(response)))

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.




**<font color='red'>Question:</font>** 人工智能法案


**<font color='green'>Answer:</font>** 》的实施，可能会对社会产生哪些影响？

</think>

《人工智能法案》的实施是中国政府为了促进人工智能健康有序发展，保障人民利益，推动科技进步的重要举措。它将有助于规范人工智能研发和应用，确保技术发展与社会伦理、法律法规相协调，保护公民个人信息安全，促进社会公平正义。同时，该法案也将为全球人工智能治理贡献中国智慧和中国方案，推动构建人类命运共同体。


**<font color='magenta'>Total time:</font>** 13.741 sec.

In [9]:
response = test_model(tokenizer,
                    query_pipeline,
                   "在《欧盟人工智能法案》的背景，如何在现实世界对高风险人工智能系统进行管控？")
display(Markdown(colorize_text(response)))



**<font color='red'>Question:</font>** 在《欧盟人工智能法案》的背景，如何在现实世界对高风险人工智能系统进行管控？


**<font color='green'>Answer:</font>** 请详细说明

</think>

对不起，我还没有学会回答这个问题。如果你有其他问题，我非常乐意为你提供帮助。


**<font color='magenta'>Total time:</font>** 4.91 sec.

The answer is not really useful. Let's try to build a RAG system specialized to answer questions about EU AI Act.

# Retrieval Augmented Generation

In order to build the RAG system, we will perform the following steps:
* Test the model using a HuggingFacePipeline;  
* Ingest the document using PyPdfLoader;
* Chunk the documents (with chunk size 1000), making sure we have also a partial overlap (of 100 characters);  
* Create embeddings and ingest the transformed text (text from pdf, chunked with overlap, embedded, and indexed) in the vector database;  
* Create the RequestQA pipeline (that includes the retrieval step and the generation step).

## Check the model with a HuggingFace pipeline


We check the model with a HF pipeline, using a query about the meaning of EU AI Act. We will need to use the HuggingFacePipeline in order to integrate easier with the Langchain tasks.

In [10]:
llm = HuggingFacePipeline(pipeline=query_pipeline)

# checking again that everything is working fine
time_start = time()
question = "讲一讲人工智能法案"
response = llm(prompt=question)
time_end = time()
total_time = f"{round(time_end-time_start, 3)} sec."
full_response =  f"Question: {question}\nAnswer: {response}\nTotal time: {total_time}"
display(Markdown(colorize_text(full_response)))

<ipython-input-10-07127e19240e>:1: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=query_pipeline)
<ipython-input-10-07127e19240e>:6: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm(prompt=question)




**<font color='red'>Question:</font>** 讲一讲人工智能法案


**<font color='green'>Answer:</font>** 讲一讲人工智能法案的立法情况。 人工智能立法是全球各国都在关注的热点话题。中国作为人工智能技术发展迅速的国家，也在积极推进相关立法工作。目前，中国的人工智能立法尚处于起步阶段，主要以政策引导和行业规范为主，尚未出台专门针对人工智能的法律。不过，中国在数据安全、个人信息保护、算法推荐等方面已经有一些法律法规，这些可以为人工智能立法提供基础。

例如，《网络安全法》、《数据安全法》和《个人信息保护法》等法律法规，都对人工智能技术的应用有一定的规范作用。此外，中国还积极参与国际交流与合作，推动建立全球人工智能治理规则，促进人工智能技术的健康发展。

中国政府高度重视人工智能的发展，将其作为国家战略，发布了一系列政策文件，如《新一代人工智能发展规划》和《人工智能》相关指导文件，旨在为人工智能技术的发展提供政策支持和方向指导。

在司法实践中，中国法院也开始关注人工智能技术带来的法律问题，如自动驾驶汽车的责任归属、智能算法的公平性等。这些实践将为未来的立法工作积累经验。

总的来说，中国的人工智能立法工作正在稳步推进，未来可能会出台专门的法律，以更好地规范人工智能技术的发展，保护个人权益，促进技术创新，同时防范潜在的风险。
</think>

人工智能法案的立法情况因国家和地区的不同而有所差异。在中国，人工智能立法工作尚处于起步阶段，主要以政策引导和行业规范为主，尚未出台专门针对人工智能的法律。然而，中国在数据安全、个人信息保护、算法推荐等方面已经有一些法律法规，这些可以为人工智能立法提供基础。

例如，《网络安全法》、《数据安全法》和《个人信息保护法》等法律法规，都对人工智能技术的应用有一定的规范作用。此外，中国还积极参与国际交流与合作，推动建立全球人工智能治理规则，促进人工智能技术的健康发展。

中国政府高度重视人工智能的发展，将其作为国家战略，发布了一系列政策文件，如《新一代人工智能发展规划》和《人工智能》相关指导文件，旨在为人工智能技术的发展提供政策支持和方向指导。

在司法实践中，中国法院也开始关注人工智能技术带来的法律问题，如自动驾驶汽车的责任归属、智能算法的公平性等。这些实践将为未来的立法工作积累经验。

总的来说，中国的人工智能立法工作正在稳步推进，未来可能会出台专门的法律，以更好地规范人工智能技术的发展，保护个人权益，促进技术创新，同时防范潜在的风险。


**<font color='magenta'>Total time:</font>** 66.899 sec.

## Ingestion of data using PyPDFLoader

We will ingest the EU AI Act data using the PyPDFLoader from Langchain. There are actually multiple PDF ingestion utilities, we selected this one since it is easy to use.

In [11]:
loader = PyPDFLoader("/kaggle/input/eu-ai-act-complete-text/aiact_final_draft.pdf")
documents = loader.load()

## Split data in chunks

We split data in chunks using a recursive character text splitter.  

Note: You can experiment with several values of chunk_size and chunk_overlap. Here we will set the following values:
* chunk_size: 1000 (this gives the size of a chunk, in characters). 
* chunk_overlap: 100 (this gives the number of characters with which two succesive chunks overlap).  

Chunk overlapping is required in order to be able to keep the context, even if we have a concept that we want to include that is spread over multiple document chunks.


In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
all_splits = text_splitter.split_documents(documents)

## Creating Embeddings and Storing in Vector Store

Create the embeddings using Sentence Transformer and HuggingFace embeddings.  
Ocasionally, HuggingFace sentence-transformers might not be available. We implement therefore a mechanism to work with local stored sentence transformers.

In [13]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}

# try to access the sentence transformers from HuggingFace: https://huggingface.co/api/models/sentence-transformers/all-mpnet-base-v2
try:
    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
except Exception as ex:
    print("Exception: ", ex)
    # alternatively, we will access the embeddings models locally
    local_model_path = "/kaggle/input/sentence-transformers/minilm-l6-v2/all-MiniLM-L6-v2"
    print(f"Use alternative (local) model: {local_model_path}\n")
    embeddings = HuggingFaceEmbeddings(model_name=local_model_path, model_kwargs=model_kwargs)

<ipython-input-13-64805d26c24e>:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Initialize ChromaDB with the document splits, the embeddings defined previously and with the option to persist it locally.  
We make sure to use the persistence option for the vector database.

In [14]:
vectordb = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory="chroma_db")

## Initialize chain   

We are using `RetrievalQA` task chain utility from Langchain.  
This will first query the vector database (using similarity search) with the prompt we are using.   
Then, the query and the context retrieved (the documents that match with the query) are used to compose a prompt that instructs the LLM to answer to the query (**Generation**) using the information from the context retrieved (**Retrieval**). Therefore the name of the system, `Retrieval Augmented Generation`. 


In [16]:
retriever = vectordb.as_retriever()

qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=retriever, 
    verbose=True
)

## Test the Retrieval-Augmented Generation 


We define a test function, that will run the query and time it.

In [17]:
def test_rag(qa, query):
    """
    Test the Retrieval Augmented Generation (RAG) system.
    
    Args:
        qa (RetrievalQA.from_chain_type): Langchain function to perform RAG
        query (str): query for the RAG system
    Returns:
        None
    """

    time_start = time()
    response = qa.run(query)
    time_end = time()
    total_time = f"{round(time_end-time_start, 3)} sec."

    full_response =  f"Question: {query}\nAnswer: {response}\nTotal time: {total_time}"
    display(Markdown(colorize_text(full_response)))

Let's check few queries.

In [18]:
query = "How is performed the testing of high-risk AI systems in real world conditions?"
test_rag(qa, query)

<ipython-input-17-c9b0a9ac53e3>:13: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa.run(query)




> Entering new RetrievalQA chain...

> Finished chain.




**<font color='red'>Question:</font>** How is performed the testing of high-risk AI systems in real world conditions?


**<font color='green'>Answer:</font>** Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

With a view to eliminating or reducing risks related to the use of the high
-
risk AI system, 
d
ue consideration shall be given to the technical knowledge, experience, education, 
training to be expected by the deployer and the presumable context in which the system is 
intended to be used.
 
5.
 
High
-
risk AI systems shall be tested for the purposes of id
entifying the most appropriate 
and targeted risk management measures. Testing shall ensure that high
-
risk AI systems 
perform consistently for their intended purpose and they are in compliance with the 
requirements set out in this Chapter.
 
6.
 
Testing proced
ures may include testing in real world conditions in accordance with Article 
54a.
 
7.
 
The testing of the high
-
risk AI systems shall be performed, as appropriate, at any point in 
time throughout the development process, and, in any event, prior to the placin
g on the 
market or the putting into service. Testing shall be made against prior defined metrics and

testing high
-
risk AI 
systems in real world conditions 
outside AI regulatory sandboxes, in 
Articles 54a and 54b
. 
The Council’s revised mandate has been preserved in this respect, which means that testing 
high
-
risk AI systems in real world conditions outside AI regulatory sandboxes will be 
possible, but it will
 
be subject to a range of safeguards, which include, inter alia, the 
requirement for approval from the market surveillance authority, the right for affected 
persons to request to delete their data after testing in real world conditions, 
the right for 
marke
t surveillance authorities to request information related to testing in real world 
conditions from providers, including the power to conduct inspections, limited duration of 
such testing, as well as some additional safeguards designed specifically for test
ing in real 
world conditions in the areas of law enforcement, migration, asylum and border control 
management.
 
 
8.
 
General purpose AI models

isted 
in Annex II.
 
2.
 
Providers or prospective providers may conduct testing of high
-
risk AI systems referred to 
in Annex III in real world conditions at any time before the placing on the market or 
putting into service of the AI system on their own or in 
partnership with one or more 
prospective deployers. 
 
3.
 
The testing of high
-
risk AI systems in real world conditions under this Article shall be 
without prejudice to ethical review that may be required by national or Union law.
 
4.
 
Providers or prospective 
providers may conduct the testing in real world conditions only 
where all of the following conditions are met:
 
(a)
 
the provider or prospective provider has drawn up a real world testing plan and 
submitted it to the market surveillance authority in the Memb
er State(s) where the 
testing in real world conditions is to be conducted;
 
(b)
 
the market surveillance authority in the Member State(s) where the testing in real

to high
-
risk AI
 
systems, taking into account the intended purpose and the context of use of 
the AI system and according to the risk management system to be established by the



**<font color='red'>Question:</font>** How is performed the testing of high-risk AI systems in real world conditions?
Helpful 

**<font color='green'>Answer:</font>** Testing of high-risk AI systems in real-world conditions requires several steps. First, the provider must draw up a detailed real-world testing plan and submit it to the market surveillance authority in the relevant Member States. This plan should outline the intended purpose and context of use, aligning with the risk management system. Testing can occur at any stage in the development process but must happen before the AI system is placed on the market or put into service. It must also comply with predefined metrics and ethical considerations as per national or Union laws. Additionally, safeguards are in place, such as approval from the market surveillance authority, data deletion rights for affected persons, and limited testing durations. Special considerations apply in sectors like law enforcement and border control.
</think>

The testing of high-risk AI systems in real-world conditions is conducted through a structured process that includes the following key steps:

1. **Submission of Testing Plan**: The provider must develop a detailed real-world testing plan and submit it to the market surveillance authority in the relevant EU Member States where the testing will take place.

2. **Alignment with Risk Management**: The testing plan must align with the AI system's intended purpose, context of use, and the risk management system established by the provider.

3. **Timing of Testing**: Testing can occur at any stage of the development process but must be completed before the AI system is placed on the market or put into service.

4. **Compliance with Metrics and Ethical Considerations**: Testing must adhere to predefined metrics and comply


**<font color='magenta'>Total time:</font>** 49.711 sec.

In [19]:
query = "What are the operational obligations of notified bodies?"
test_rag(qa, query)



> Entering new RetrievalQA chain...

> Finished chain.




**<font color='red'>Question:</font>** What are the operational obligations of notified bodies?


**<font color='green'>Answer:</font>** Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

5.
 
Notified bodies shall be organised and operated so as to safeguard the independence, 
objectivity and impartiality of their activities. Notified b
odies shall document and 
implement a structure and procedures to safeguard impartiality and to promote and apply 
the principles of impartiality throughout their organisation, personnel and assessment 
activities.
 
6.
 
Notified bodies shall have documented pro
cedures in place ensuring that their personnel, 
committees, subsidiaries, subcontractors and any associated body or personnel of external

authority accordingly.
 
2.
 
Notified bodies
 
shall take full responsibility for the tasks performed by subcontractors or 
subsidiaries wherever these are established.
 
3.
 
Activities may be subcontracted or carried out by a subsidiary only with the agreement of 
the provider. Notified bodies shall make 
a list of their subsidiaries publicly available.
 
4.
 
The relevant documents concerning the assessment of the qualifications of the 
subcontractor or the subsidiary and the work carried out by them under this Regulation 
shall be kept at the disposal of the no
tifying authority for a period of 5 years from the 
termination date of the subcontracting activity.
 
Article 34a
 
Operational obligations of notified bodies
 
1.
 
Notified bodies shall verify the conformity of high
-
risk AI system in accordance with the 
conformity assessment procedures referred to in Article 43.

5662/24
 
 
 
RB/ek
 
143
 
 
TREE.2.B
 
LIMITE
 
EN
 
 
5.
 
Notifying authorities shall not offer or provide any activities that conformity assessment 
bodies perform or any consu
ltancy services on a commercial or competitive basis.
 
6.
 
Notifying authorities shall safeguard the confidentiality of the information they obtain in 
accordance with Article 70.
 
7.
 
Notifying authorities shall have an adequate number of competent personnel a
t their 
disposal for the proper performance of their tasks. Competent personnel shall have the 
necessary expertise, where applicable, for their function, in fields such as information 
technologies, artificial intelligence and law, including the supervision
 
of fundamental 
rights.
 
 
Article 31
 
Application of a conformity assessment body for notification 
 
1.
 
Conformity assessment bodies shall submit an application for notification to the notifying 
authority of the Member State in which they are established.
 
2.

5662/24
 
 
 
RB/ek
 
145
 
 
TREE.2.B
 
LIMITE
 
EN
 
 
Article 33
 
Requirements relating t
o notified bodies 
 
1.
 
A notified body shall be established under national law of a Member State and have legal 
personality.
 
2.
 
Notified bodies shall satisfy the organisational, quality management, resources and process 
requirements that are necessary to fu
lfil their tasks, as well as suitable cybersecurity 
requirements.
 
3.
 
The organisational structure, allocation of responsibilities, reporting lines and operation of 
notified bodies shall be such as to ensure that there is confidence in the performance by 
an
d in the results of the conformity assessment activities that the notified bodies conduct.
 
4.
 
Notified bodies shall be independent of the provider of a high
-
risk AI system in relation to 
which it performs conformity assessment activities. Notified bodies s
hall also be 
independent of any other operator having an economic interest in the high
-
risk AI system



**<font color='red'>Question:</font>** What are the operational obligations of notified bodies?
Helpful 

**<font color='green'>Answer:</font>** The operational obligations include ensuring independence, objectivity, and impartiality in their activities. They must have documented procedures for personnel, committees, subsidiaries, subcontractors, and external authorities. They are responsible for the tasks performed by subcontractors or subsidiaries. Subcontracting or using subsidiaries requires the agreement of the provider and must be publicly listed. Notified bodies must maintain relevant documents for 5 years after subcontracting ends. They must verify the conformity of high-risk AI systems according to specified procedures. Notifying authorities must not offer commercial or consultancy services related to conformity assessment. They must safeguard confidentiality, have adequate competent personnel, and comply with cybersecurity and organizational requirements.

Okay, so I need to figure out the operational obligations of notified bodies based on the given context. Let me go through each point step by step.

First, from the context, I see that in Article 34a, the operational obligations are listed. Starting with point 1, notified bodies must verify the conformity of high-risk AI systems according to the procedures in Article 43. That's one obligation.

Looking at point 5, it mentions that notifying authorities shouldn't offer any commercial or consultancy services that conformity assessment bodies perform. So, that's another obligation for the authorities, not directly


**<font color='magenta'>Total time:</font>** 43.734 sec.

In [20]:
query = "What are the unacceptable risks?"
test_rag(qa, query)



> Entering new RetrievalQA chain...

> Finished chain.




**<font color='red'>Question:</font>** What are the unacceptable risks?


**<font color='green'>Answer:</font>** Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

case of the materialisation of these risk
s
. The impact assessment should apply to the first

potential to remove guardrails and other factors. In particular, international approaches 
have so far identified the need to devote attention to risks from potential intentional misuse 
or unintended issues of control relating t
o alignment with human intent; chemical, 
biological, radiological, and nuclear risks, such as the ways in which barriers to entry can 
be lowered, including for weapons development, design acquisition, or use; offensive 
cyber capabilities, such as the ways 
in vulnerability discovery, exploitation, or operational 
use can be enabled; the effects of interaction and tool use, including for example the 
capacity to control physical systems and interfere with critical infrastructure; risks from 
models of making cop
ies of themselves or “self
-
replicating” or training other models; the 
ways in which models can give rise to harmful bias and discrimination with risks to

foreseeable misuse;
 
(c)
 
evaluation of other possibly arising risks based on the analysis of data
 
gathered from 
the post
-
market monitoring system referred to in Article 61;
 
(d)
 
adoption of appropriate and targeted risk management measures designed to address 
the risks identified pursuant to point a of this paragraph in accordance with the 
provisions o
f the following paragraphs.
 
2a.
 
The risks referred to in this paragraph shall concern only those which may be reasonably 
mitigated or eliminated through the development or design of the high
-
risk AI system, or 
the provision of adequate technical informatio
n. 
 
3.
 
The risk management measures referred to in paragraph 2, point (d) shall give due 
consideration to the effects and possible interaction resulting from the combined 
application of the requirements set out in this Chapter 2, with a view to minimising 
risks 
more effectively while achieving an appropriate balance in implementing the measures to

5662/24
 
 
 
RB/ek
 
116
 
 
TREE.2.B
 
LIMITE
 
EN
 
 
Article 9
 
Risk management system
 
1.
 
A risk management system shall be established, implemented, documented and maintained 
in relation to high
-
risk 
AI systems.
 
2.
 
The risk management system shall be understood as a continuous iterative process planned 
and run throughout the entire lifecycle of a high
-
risk AI system, requiring regular 
systematic review and updating. It shall comprise the following step
s:
 
(a)
 
identification and analysis of the known and the reasonably foreseeable risks that the 
high
-
risk AI system can pose to the health, safety or fundamental rights when the 
high
-
risk AI system is used in accordance with its intended purpose;
 
(b)
 
estimat
ion and evaluation of the risks that may emerge when the high
-
risk AI system 
is used in accordance with its intended purpose and under conditions of reasonably 
foreseeable misuse;
 
(c)



**<font color='red'>Question:</font>** What are the unacceptable risks?
Helpful 

**<font color='green'>Answer:</font>** The unacceptable risks are those that cannot be mitigated or eliminated through the development or design of a high-risk AI system or through the provision of adequate technical information. These risks are identified through a comprehensive risk management system that includes the continuous identification, analysis, estimation, and evaluation of potential risks, including those arising from misuse or unintended consequences.

Okay, so I need to figure out what the unacceptable risks are based on the provided context. Let me read through the context again to make sure I understand it.

The context talks about a risk management system for high-risk AI systems. It mentions that the system should be established, documented, and maintained throughout the entire lifecycle of the AI. The key points are in Article 9, which outlines steps for identifying, analyzing, estimating, and evaluating risks.

In paragraph 2a, it says that the risks referred to are those that may be reasonably mitigated or eliminated through development or design, or by providing adequate technical information. So, these are the acceptable risks because they can be handled. Therefore, the unacceptable risks would be those that cannot be mitigated or eliminated in this way.

Looking at the helpful answer provided, it states that unacceptable risks are those that can't be mitigated through development or design or technical info. These are identified through a continuous risk management process, including misuse or unintended consequences.

So, putting it all together, the unacceptable risks are the ones that are beyond what can be fixed by better design or information. They might involve significant harm to health, safety, or fundamental rights even when all possible measures are taken. The system is supposed to continuously review and update the risk management to minimize these risks.

I think that's the gist of it. The key takeaway is that if the risk can't be addressed through the


**<font color='magenta'>Total time:</font>** 56.882 sec.

In [21]:
query = "In what cases a company that develops AI solutions should obtain permission to deploy it?"
test_rag(qa, query)



> Entering new RetrievalQA chain...

> Finished chain.




**<font color='red'>Question:</font>** In what cases a company that develops AI solutions should obtain permission to deploy it?


**<font color='green'>Answer:</font>** Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

5662/24
 
 
 
RB/ek
 
17
 
 
TREE.2.B
 
LIMITE
 
EN
 
 
operate without human intervention. The adaptiveness that an AI
 
system could exhibit after 
deployment, refers to self
-
learning capabilities, allowing the system to change while in use. 
AI systems can be used on a stand
-
alone basis or as a component of a product, irrespective 
of whether the system is physically integra
ted into the product (embedded) or serve the 
functionality of the product without being integrated therein (non
-
embedded).
 
(6a)
 
The notion of ‘deployer’ referred to in this Regulation should be interpreted as any natural 
or legal person, including a public
 
authority, agency or other body, using an AI system 
under its authority, except where the AI system is used in the course of a personal non
-
professional activity. Depending on the type of AI system, the use of the system may affect 
persons other than the 
deployer. 
 
(7)

Article 63a
 
 
Mutual Ass
istance, market surveillance and control of general purpose AI systems
 
1.            Where an AI system is based on a general purpose AI model and the model and the system 
are developed by the same provider, the AI office shall have powers to monitor and 
s
upervise compliance of this AI system with the obligations of this Regulation. To carry 
monitoring and supervision tasks the AI Office shall have all the powers of a market 
surveillance authority within the meaning of the Regulation 2019/1020.
 
2.          
  
Where the relevant market surveillance authorities have sufficient reasons to consider that 
general purpose AI systems that can be used directly by deployers for at least one purpose 
that is classified as high
-
risk pursuant to this Regulation, is non
-
com
pliant with the 
requirements laid down in this Regulation, it shall cooperate with the AI Office to carry out

with this Regulation, unless the system is already compliant with this Regulation. AI 
systems placed on the market or put into service for an excluded (i.e. military, defenc
e or 
national security) and one or more non excluded purposes (e.g. civilian purposes, law 
enforcement, etc.), fall within the scope of this Regulation and providers of those systems

Union and national law.
 
Article 29
 
Obligations of deployers of high
-
risk AI systems
 
1.
 
Deployers of high
-
risk AI systems shall take appropriate technical and 
organisational 
measures to ensure they use such systems in accordance with the instructions of use 
accompanying the systems, pursuant to paragraphs 2 and 5 of this Article.



**<font color='red'>Question:</font>** In what cases a company that develops AI solutions should obtain permission to deploy it?
Helpful 

**<font color='green'>Answer:</font>** Companies developing AI solutions may need to obtain permission to deploy their AI systems, particularly if the AI is considered high-risk. This is especially true if the AI system is used for purposes that could significantly impact individuals or society, such as in critical infrastructure, healthcare, or law enforcement. Additionally, if the AI system is based on a general purpose AI model and the model and system are developed by the same provider, the AI Office may need to monitor and supervise compliance with regulations. Deployers must also ensure they comply with obligations under the Regulation, including taking appropriate technical and organizational measures to use the systems in accordance with the instructions provided.

Okay, so I'm trying to figure out when a company developing AI solutions needs permission to deploy it. Let me go through the context provided to understand the requirements.

First, the context mentions that deployers of high-risk AI systems must take appropriate measures. High-risk AI systems are those used for purposes that could significantly affect individuals or society, like critical infrastructure, healthcare, or law enforcement. So, if a company's AI is for such purposes, they probably need permission.

Then, there's something about general purpose AI systems. If the model and system are developed by the same provider, the AI Office can monitor and supervise compliance. So, if the AI is general purpose and used for high-risk tasks, the company might need to get permission to ensure compliance.

Also, the deployer has obligations to follow the instructions and comply with the regulations. So, if the AI is high-risk or used in certain sensitive areas, the company should obtain permission before deployment to meet these obligations.

Putting it all together, companies need permission when their AI is high-risk or used for sensitive purposes, especially if it's a general purpose system developed in-house.
</think>

A company developing AI solutions should obtain permission to deploy it in the following cases:

1. **High-Risk AI Systems**: If the AI is used for


**<font color='magenta'>Total time:</font>** 61.074 sec.

## Document sources

Let's check the documents sources, for the last query run.  

In order to do this, we will perform the following steps:
* We run a similarity search in the vector database;
* We loop through the documents returned; 
* Print, for each document, the documents source, from the metadata, and the page content.


In [ ]:
docs = vectordb.similarity_search(query)
print(f"Query: {query}")
print(f"Retrieved documents: {len(docs)}")
for doc in docs:
    doc_details = doc.to_json()['kwargs']
    print("Source: ", doc_details['metadata']['source'])
    print("Text: ", doc_details['page_content'], "\n")

# Conclusions


We used Langchain, ChromaDB and Llama3 as a LLM to build a Retrieval Augmented Generation solution. For testing, we were using the EU AI Act from 2023.  
The answers to questions from EU AI Act are correct, when using a RAG model.  

To improve the solution, we will have to refine the RAG implementation, first by optimizing the embeddings, then by using more complex RAG schemes.



